# Ghost Band Research

This notebook replays the V2 VWAP engine on historical US100 M1 data to study how the Gaussian ghost bands behave alongside the solid bands.

## 1. Load data and choose a launch time

Use the 30 completed candles before the simulated launch to seed VWAP. Load up to 90 minutes before that anchor to warm up the solid band width.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.config import CONFIG
from src.engine import EngineState, update_engine_state
from src.loaders import load_mt5_csv
from src.replay import run_replay

ANCHOR_BARS = 30
WARMUP_MINUTES = 90
REPLAY_MINUTES = 60
LAUNCH = pd.Timestamp("2026-01-30 16:10", tz="UTC")

bars = load_mt5_csv(str(CONFIG["data_dir"] / CONFIG["csv_filename"]))
bars = bars.drop_duplicates(subset="datetime")

seed = bars.loc[bars["datetime"] < LAUNCH].tail(ANCHOR_BARS)
assert len(seed) == ANCHOR_BARS, "Insufficient completed bars before launch."

anchor = seed["datetime"].iloc[0]
warmup_start = anchor - pd.Timedelta(minutes=WARMUP_MINUTES)
replay_end = LAUNCH + pd.Timedelta(minutes=REPLAY_MINUTES)

replay_bars = bars.loc[
    bars["datetime"].ge(warmup_start) & bars["datetime"].lt(replay_end)
]
config = {**CONFIG, "vwap_anchor_utc": anchor}

## 2. Replay the engine and verify the anchor

Process the candles in order and record the VWAP and bands after each close. Check the starting VWAP against a direct tick-volume-weighted average of the 30 seed candles.

In [ ]:
probability_table = pd.DataFrame(columns=["zone"])
rows = []

for state in run_replay(
    replay_bars,
    config,
    probability_table,
    EngineState=EngineState,
    update_engine_state=update_engine_state,
    speed=100_000,
):
    rows.append({
        "datetime": state.datetime,
        "close": state.close,
        "vwap": state.reference,
        "session_bars": state.session_bar_count,
        **{f"solid_{key}": value for key, value in state.bands.items()},
        **{f"ghost_{key}": value for key, value in state.ghost_bands.items()},
    })

snapshots = pd.DataFrame(rows).set_index("datetime")
seed_done = snapshots.loc[seed["datetime"].iloc[-1]]
expected_vwap = np.average(seed["typical_price"], weights=seed["tick_volume"])

assert seed_done["session_bars"] == ANCHOR_BARS
assert np.isclose(seed_done["vwap"], expected_vwap, rtol=0, atol=1e-8)

print(f"VWAP anchor: {anchor} | Completed seed bars: {ANCHOR_BARS}")
print(f"VWAP at launch: {seed_done['vwap']:.2f}")

## 3. Plot the levels available at each candle open

Pair each candle with the VWAP and band levels calculated at the previous close. This is the timing we'll use to test whether a wick reached a band that was already visible.

In [ ]:
level_columns = snapshots.filter(regex=r"^(vwap$|solid_|ghost_)").columns
chart = snapshots.copy()
chart[level_columns] = snapshots[level_columns].shift(1)
chart = chart.loc[chart.index >= LAUNCH]

assert len(chart) == REPLAY_MINUTES
assert chart["vwap"].iloc[0] == seed_done["vwap"]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(chart.index, chart["close"], color="black", label="M1 close")
ax.step(chart.index, chart["vwap"], where="post", color="tab:blue", label="VWAP")

colors = ["#1b8f56", "#db8519", "#bd3549"]
for level, color in enumerate(colors, start=1):
    for sign in ("+", "-"):
        ax.step(chart.index, chart[f"solid_{level}{sign}"], where="post", color=color)
        ax.step(
            chart.index,
            chart[f"ghost_{level}{sign}"],
            where="post",
            color=color,
            linestyle="--",
        )

ax.set_title("US100 M1 · levels available at candle open · dashed lines are ghost bands")
ax.set_ylabel("Price")
ax.legend()
fig.tight_layout()
plt.show()